# Overture Maps: Fetching Land Cover Data

**CCOM 6994: Data Analysis Tools - Final Project**

This notebook fetches **Land Cover** data from the **Overture Maps Foundation** dataset
using the [`overturemaestro`](https://kraina-ai.github.io/overturemaestro/) library for
GeoPandas-friendly data access with built-in caching.

---

## 🌍 What is Land Use-Land Cover (LULC)?

**Land Cover** refers to the physical and biological cover of the Earth's surface,
including vegetation, water, bare soil, and artificial structures. It answers the
question: *"What is physically present on the ground?"*

### Common Land Cover Classes
| Class | Description | Examples |
|-------|-------------|----------|
| **Forest** | Tree-dominated areas | Deciduous, coniferous, mixed forests |
| **Shrub** | Woody vegetation <5m tall | Chaparral, scrubland |
| **Grass** | Herbaceous vegetation | Prairies, pastures, lawns |
| **Crop** | Agricultural land | Farms, orchards, vineyards |
| **Urban** | Built-up areas | Cities, roads, parking lots |
| **Barren** | Minimal vegetation | Deserts, rock outcrops, beaches |
| **Wetland** | Water-saturated areas | Marshes, swamps, bogs |
| **Water** | Open water bodies | Lakes, rivers, oceans |

### How is Land Cover Data Derived?

Land cover maps are typically derived from:

1. **Satellite Imagery**: Landsat, Sentinel-2, MODIS provide multispectral data
2. **Machine Learning Classification**: Random forests, neural networks classify pixels
3. **Ground Truth Validation**: Field surveys verify accuracy
4. **Temporal Analysis**: Multi-date imagery captures seasonal changes

Popular land cover datasets include:
- **NLCD** (National Land Cover Database) - 30m resolution, USA
- **ESA WorldCover** - 10m resolution, global
- **Copernicus CORINE** - 100m resolution, Europe
- **Overture Maps** - Vector-based, [derived from ESA's 2020 WorldCover](https://docs.overturemaps.org/blog/2024/05/16/land-cover/) that uses Sentinel-2 Imagery

---

## 🎯 Why Land Cover Matters for Solar PV Analysis

Understanding land cover context around solar installations helps us:

1. **Site Characterization**: What land types host solar farms?
2. **Land Use Change**: Are panels replacing cropland, forest, or developed areas?
3. **Environmental Impact**: Assess ecological footprint of solar development
4. **Policy Analysis**: Identify patterns in permitting across land types
5. **Predictive Modeling**: Which land cover types are most likely to have future solar?

---

## 📊 Overture Maps Land Cover Schema

Overture Maps provides vector-based land cover with the following structure:

| Column | Type | Description |
|--------|------|-------------|
| `id` | VARCHAR | Unique feature identifier |
| `geometry` | POLYGON | Feature boundary |
| `subtype` | VARCHAR | Land cover class (forest, urban, crop, etc.) |
| `cartography` | STRUCT | Rendering hints (min_zoom, max_zoom) |
| `sources` | ARRAY | Data provenance |

**Available subtypes**: `barren`, `crop`, `forest`, `grass`, `mangrove`, `moss`, 
`shrub`, `snow`, `urban`, `wetland`

---

## 🔧 Setup: Import Libraries

In [1]:
import os
import time
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import duckdb
import pandas as pd
import geopandas as gpd
from shapely import wkt, wkb
import matplotlib.pyplot as plt

# OvertureMaestro for Overture Maps data fetching
import overturemaestro as om
from overturemaestro.advanced_functions.wide_form import _get_all_possible_column_names

# Census geometry reader
import censusdis.maps as cem

from dotenv import load_dotenv
from tqdm.auto import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='Geometry column does not contain geometry')

# Load environment variables
load_dotenv(dotenv_path=Path('../.env'))

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(f"✅ OvertureMaestro version: {om.__version__}")
print(f"✅ Latest Overture release: {om.get_newest_release_version()}")

✅ OvertureMaestro version: 0.5.0
✅ Latest Overture release: 2025-11-19.0


## ⚙️ Configuration

Adjust these settings based on your hardware and processing needs.

In [2]:
# === PROCESSING CONFIGURATION ===

# Database path (relative to notebooks/ directory)
DB_PATH = os.getenv('PROJECT_DB', '../db/pv_project.ddb')

# Overture Maps parameters
OVERTURE_RELEASE = os.getenv('OVERTURE_RELEASE', '2025-11-19.0')


# Parallel processing settings
# Set based on your hardware: 16 cores = try 8-12 workers
# Too many workers can cause memory issues or API throttling
MAX_WORKERS = int(os.getenv('MAX_WORKERS', '8'))

# States to process (None = all states with PV installations)
# For testing, use a subset: ['CA', 'AZ', 'NV']
STATES_TO_PROCESS = None  

# Save interval: commit to DB every N counties to avoid data loss
SAVE_INTERVAL = 10

# DuckDB thread configuration (for multi-threaded operations)
DUCKDB_THREADS = int(os.getenv('DUCKDB_THREADS', '8'))

print(f"📊 Configuration:")
print(f"   Database: {DB_PATH}")
print(f"   Max parallel workers: {MAX_WORKERS}")
print(f"   DuckDB threads: {DUCKDB_THREADS}")
print(f"   States filter: {STATES_TO_PROCESS or 'All'}")

📊 Configuration:
   Database: ../db/pv_project.ddb
   Max parallel workers: 14
   DuckDB threads: 16
   States filter: All


## 📥 Task 1: Load Census-Enriched PV Data

We load our PV dataset that has been enriched with Census identifiers in notebook 02.
This gives us the list of counties that contain solar installations.

In [3]:
print(f"📂 Connecting to database: {DB_PATH}")

con = duckdb.connect(DB_PATH, read_only=True)
con.execute("INSTALL spatial; LOAD spatial;")
con.execute(f"SET threads={DUCKDB_THREADS};")

# List available tables
tables = con.execute("SHOW TABLES").fetchall()
print("📋 Available tables:")
for t in tables:
    try:
        count = con.execute(f"SELECT COUNT(*) FROM \"{t[0]}\"").fetchone()[0]
        print(f"   - {t[0]}: {count:,} rows")
    except:
        print(f"   - {t[0]}: (error reading)")

📂 Connecting to database: ../db/pv_project.ddb
📋 Available tables:
   - census_enriched_pv_data: 41,196 rows
   - overture_land_cover: 37,685 rows
   - processed_pv_data: 41,867 rows


In [4]:
# Load the census-enriched PV data
print("\n📥 Loading census_enriched_pv_data...")
pv_df = con.execute("SELECT ST_AsText(geometry) as geometry, * EXCLUDE (geometry) FROM census_enriched_pv_data").df()
pv_df['geometry'] = pv_df['geometry'].apply(wkt.loads)
pv_gdf = gpd.GeoDataFrame(pv_df, geometry='geometry', crs='EPSG:4326')

print(f"✅ Loaded {len(pv_gdf):,} PV installations")
print(f"   States: {pv_gdf['STATE_ABBR'].nunique()}")
print(f"   Counties: {pv_gdf['COUNTY_GEOID'].nunique()}")

con.close()


📥 Loading census_enriched_pv_data...
✅ Loaded 41,196 PV installations
   States: 54
   Counties: 1071


In [5]:
pv_gdf.geometry.dtype

## 🗺️ Task 2: Get County Geometries

We fetch actual county polygon geometries from Census Bureau shapefiles.
These polygons are used as precise filters when querying Overture Maps - 
much more accurate than simple bounding boxes.

In [6]:
# Get unique counties from our PV data
pv_counties_info = pv_gdf[['STATE_FIPS', 'COUNTY_FIPS', 'COUNTY_GEOID', 'COUNTY_NAME', 'STATE_ABBR']].drop_duplicates()
print(f"📊 PV installations span {len(pv_counties_info)} unique counties")

# Show top states by county count
print(f"\n📊 Counties with PV by state (top 10):")
print(pv_counties_info.groupby('STATE_ABBR').size().sort_values(ascending=False).head(10))

📊 PV installations span 1071 unique counties

📊 Counties with PV by state (top 10):
STATE_ABBR
GA    87
NC    85
NY    52
CA    52
TX    47
OH    46
MN    42
CO    41
IN    36
FL    36
dtype: int64


In [7]:
# Fetch county geometries from Census Bureau
print("\n🗺️ Fetching county geometries from Census Bureau...")
reader = cem.ShapeReader(year=2020)
county_bounds_gdf = reader.read_cb_shapefile(
    shapefile_scope="us", 
    geography="county", 
    crs="EPSG:4326"
)
print(f"✅ Loaded {len(county_bounds_gdf):,} county geometries")

# Filter to only counties with PV installations
pv_county_geoids = set(pv_counties_info['COUNTY_GEOID'].tolist())
pv_county_bounds = county_bounds_gdf[county_bounds_gdf['GEOID'].isin(pv_county_geoids)].copy()
print(f"✅ Filtered to {len(pv_county_bounds):,} counties with PV installations")


🗺️ Fetching county geometries from Census Bureau...
✅ Loaded 3,234 county geometries
✅ Filtered to 1,071 counties with PV installations


## 🌲 Task 3: Explore Land Cover Schema

Before fetching, let's understand the available land cover categories.

In [8]:
print("🌲 Available Land Cover subtypes in Overture Maps:")
land_cover_df = _get_all_possible_column_names(theme="base", type="land_cover", release_version=OVERTURE_RELEASE, hierarchy_columns=['subtype'])
# display all land cover subtypes from fetched df
land_cover_df.head()

🌲 Available Land Cover subtypes in Overture Maps:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,subtype
0,barren
1,crop
2,forest
3,grass
4,mangrove


## 🧪 Task 4: Test with a Single County

Before running the full batch, let's verify the workflow with one county.

In [10]:
# Pick a test county
test_county = pv_county_bounds.iloc[0]
test_geoid = test_county['GEOID']
test_name = test_county['NAME']
test_state = test_county['STATEFP']
test_geom = test_county.geometry

# Count PV in this county
test_pv_count = len(pv_gdf[pv_gdf['COUNTY_GEOID'] == test_geoid])

print(f"🧪 Testing with: {test_name} County (GEOID: {test_geoid})")
print(f"   State FIPS: {test_state}")
print(f"   PV installations: {test_pv_count}")
print(f"   Bounds: {test_geom.bounds}")

🧪 Testing with: Bulloch County (GEOID: 13031)
   State FIPS: 13
   PV installations: 2
   Bounds: (-82.030233, 32.15289, -81.433128, 32.653385)


In [11]:
# Fetch Land Cover for test county
print(f"\n🌲 Fetching Land Cover for {test_name} County...")

t1 = time.time()
test_lc = om.convert_geometry_to_geodataframe(
    theme="base",
    type="land_cover",
    geometry_filter=test_geom,
    columns_to_download=["id", "subtype", "geometry"],
)
t2 = time.time()

print(f"✅ Fetched {len(test_lc):,} land cover features in {t2-t1:.1f}s")
print(f"\n📊 Land cover distribution:")
print(test_lc['subtype'].value_counts())


🌲 Fetching Land Cover for Bulloch County...


Finished operation in 0:00:00

✅ Fetched 4,358 land cover features in 0.0s

📊 Land cover distribution:
subtype
shrub      1859
barren      916
crop        802
urban       534
forest      182
wetland      65
Name: count, dtype: int64


## ⚡ Task 5: Parallel Batch Processing

We use Python's `ThreadPoolExecutor` to fetch data for multiple counties in parallel.
OvertureMaestro handles caching, so repeated runs are fast.

### Why ThreadPoolExecutor?
- OvertureMaestro operations are I/O-bound (network requests to S3)
- Threads work well for I/O-bound tasks (GIL is released during I/O)
- Simpler than multiprocessing, avoids serialization overhead

### Memory Considerations
- Each worker holds one county's data in memory
- With 8 workers and ~5MB per county avg, expect ~40MB concurrent usage
- We batch-save to DuckDB to limit memory growth

In [ ]:
def fetch_land_cover_for_county(county_row):
    """
    Fetch land cover data for a single county.
    Returns tuple: (county_geoid, state_fips, geodataframe or None, error or None)
    """
    geoid = county_row['GEOID']
    state_fips = county_row['STATEFP']
    geom = county_row.geometry
    
    try:
        lc_gdf = om.convert_geometry_to_geodataframe(
            theme="base",
            type="land_cover",
            geometry_filter=geom,
            columns_to_download=["id", "subtype",
        )
        
        if len(lc_gdf) > 0:
            lc_gdf = lc_gdf.reset_index()  # id becomes a column
            lc_gdf['county_geoid'] = geoid
            lc_gdf['state_fips'] = state_fips
            lc_gdf['geometry'] = lc_gdf['geometry'].apply(lambda g: g.wkt)
            return (geoid, state_fips, lc_gdf, None)
        else:
            return (geoid, state_fips, None, None)
            
    except Exception as e:
        return (geoid, state_fips, None, str(e))

In [13]:
# Filter states if configured
if STATES_TO_PROCESS:
    counties_to_process = pv_county_bounds[pv_county_bounds['STATEFP'].isin(
        [s if len(s) == 2 else f"0{s}" for s in STATES_TO_PROCESS]
    )]
else:
    counties_to_process = pv_county_bounds

print(f"🔄 Processing {len(counties_to_process)} counties with {MAX_WORKERS} parallel workers")

# Prepare county data for parallel processing
county_records = [row for _, row in counties_to_process.iterrows()]

🔄 Processing 1071 counties with 14 parallel workers


In [14]:
# Initialize results storage
all_results = []
errors = []
processed_count = 0

# Open DB connection for writing
con = duckdb.connect(DB_PATH)
con.execute("INSTALL spatial; LOAD spatial;")
con.execute(f"SET threads={DUCKDB_THREADS};")

# Create output table
con.execute("DROP TABLE IF EXISTS overture_land_cover")
con.execute("""
    CREATE TABLE overture_land_cover (
        id VARCHAR,
        geometry VARCHAR,
        subtype VARCHAR,
        county_geoid VARCHAR,
        state_fips VARCHAR
    )
""")

print("✅ Created overture_land_cover table")

✅ Created overture_land_cover table


In [ ]:
# Parallel processing with progress bar
print(f"\n🚀 Starting parallel fetch with {MAX_WORKERS} workers...")
t_start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_county = {
        executor.submit(fetch_land_cover_for_county, county): county['GEOID']
        for county in county_records
    }
    
    # Process completed tasks with progress bar
    with tqdm(total=len(county_records), desc="Fetching Land Cover") as pbar:
        for future in as_completed(future_to_county):
            geoid = future_to_county[future]
            
            try:
                county_geoid, state_fips, result_df, error = future.result()
                
                if error:
                    errors.append({'county_geoid': county_geoid, 'error': error})
                elif result_df is not None:
                    all_results.append(result_df)
                
                processed_count += 1
                
                # Periodic save to database
                if len(all_results) >= SAVE_INTERVAL:
                    batch_df = pd.concat(all_results, ignore_index=True)
                    con.execute("INSERT INTO overture_land_cover SELECT * FROM batch_df")
                    pbar.set_postfix({'saved': con.execute("SELECT COUNT(*) FROM overture_land_cover").fetchone()[0]})
                    all_results = []
                    
            except Exception as e:
                errors.append({'county_geoid': geoid, 'error': str(e)})
            
            pbar.update(1)

# Save remaining results
if all_results:
    batch_df = pd.concat(all_results, ignore_index=True)
    con.execute("INSERT INTO overture_land_cover SELECT * FROM batch_df")
    print(f"💾 Final save: {con.execute('SELECT COUNT(*) FROM overture_land_cover').fetchone()[0]} total rows453")

t_end = time.time()

Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:26 < 0:00:00 • 10.02 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:17 < 0:00:00 • 6.66 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:20 < 0:00:00 • 6.65 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 • 0:00:44 < 0:00:00 • 6.88 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 • 0:00:23 < 0:00:00 • 5.58 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 • 0:00:09 < 0:00:00 • 0 it/s
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:12 < 0:00:00 • 0 it/s
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:30 < 0:00:00 • 15.53 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 • 0:00:19 < 0:00:00 • 10.81 s/it
  Downloading parquet files (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:09 < 0:00:00 • 0 it/s
⠼ Preparing multiprocessing environment • 0:00:13
  Sorting result file by geometry 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5260/5260 • 0:00:01 < 0:00:00 • 0 it/s
  Filtering data by geometry (base/land_cover) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 • 0:00:01 < 0:00:00 • 0 it/s
  Sorting result file by geometry 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3734/3734 • 0:00:00 < 0:00:00 • 0 it/s

In [ ]:
# Processing summary
total_features = con.execute("SELECT COUNT(*) FROM overture_land_cover").fetchone()[0]
unique_counties = con.execute("SELECT COUNT(DISTINCT county_geoid) FROM overture_land_cover").fetchone()[0]

print(f"\n✅ Parallel processing complete!")
print(f"   Total time: {(t_end - t_start)/60:.1f} minutes")
print(f"   Counties processed: {processed_count}")
print(f"   Counties with data: {unique_counties}")
print(f"   Total features: {total_features:,}")
print(f"   Errors: {len(errors)}")

if errors:
    print(f"\n⚠️ Counties with errors:")
    for e in errors[:5]:
        print(f"   {e['county_geoid']}: {e['error'][:50]}...")

## 📊 Task 6: Data Summary and Verification

In [ ]:
print("📊 Land Cover Summary by Subtype:")
summary = con.execute("""
    SELECT 
        subtype, 
        COUNT(*) as feature_count,
        COUNT(DISTINCT county_geoid) as county_count
    FROM overture_land_cover 
    GROUP BY subtype 
    ORDER BY feature_count DESC
""").df()
print(summary.to_string(index=False))

In [ ]:
print("\n📊 Top 10 Counties by Feature Count:")
top_counties = con.execute("""
    SELECT 
        county_geoid,
        state_fips,
        COUNT(*) as feature_count
    FROM overture_land_cover 
    GROUP BY county_geoid, state_fips
    ORDER BY feature_count DESC
    LIMIT 10
""").df()
print(top_counties.to_string(index=False))

## 🗄️ Task 7: Create Spatial Index

Adding a spatial index improves query performance for spatial operations.

In [ ]:
print("🗄️ Creating spatial index...")
try:
    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_land_cover_geom 
        ON overture_land_cover USING RTREE (ST_GeomFromText(geometry))
    """)
    print("✅ Spatial index created")
except Exception as e:
    print(f"⚠️ Could not create spatial index: {e}")

In [ ]:
# Final verification
final_count = con.execute("SELECT COUNT(*) FROM overture_land_cover").fetchone()[0]
print(f"\n📈 Final table size: {final_count:,} land cover features")

con.close()
print("✅ Database connection closed")

## 📝 Summary

This notebook demonstrated:

1. **Land Cover Fundamentals**: Physical surface classification for Earth observation
2. **Overture Maps Integration**: Using overturemaestro for GeoPandas-friendly access
3. **Parallel Processing**: ThreadPoolExecutor for I/O-bound operations
4. **Efficient Storage**: Batch saves to DuckDB with spatial indexing

### Performance Notes

| Configuration | Expected Time |
|---------------|---------------|
| Sequential (1 worker) | ~6 hours for 1,071 counties |
| 4 workers | ~1.5 hours |
| 8 workers | ~45-60 minutes |
| 16 workers | ~30-45 minutes (may hit API limits) |

**Tip**: OvertureMaestro caches downloaded data, so re-runs are much faster!

### Next Steps
→ Continue to **Notebook 05** for Land Use data
→ Use the data in analysis notebooks for PV-LULC correlation studies